<a href="https://colab.research.google.com/github/cassiecinzori/ECON3916/blob/main/Project/ECON3916_Project_Phase3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 1 | Phase 3: The Identification Strategy

### Cassandra Cinzori

## Environment and Data

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Load and subset to working women only
url = "https://vincentarelbundock.github.io/Rdatasets/csv/Ecdat/Mroz.csv"
df = pd.read_csv(url)
df_work = df[df['work'] == 'yes'].copy()

# Create log wage
df_work['log_hearnw'] = np.log(df_work['hearnw'])

# Convert city to integer dummy (1 = urban, 0 = rural)
df_work['city'] = df_work['city'].map({'yes': 1, 'no': 0})

print(f"Working sample: {len(df_work)} observations")

Working sample: 428 observations


##  STEP 3.1: Functional Form Justification

#### Functional Form: Log-Level (Mincer Wage Equation)

The dependent variable is log(hearnw) rather than hearnw in levels because:

1. **Log-normality**: Wage distributions are right-skewed in levels but
   approximately normal after log transformation (confirmed in Phase 2).
2. **Interpretability**: In a Log-Level model, the coefficient on educw
   represents the approximate percentage change in wages for each
   additional year of education — directly mapping onto human capital theory.
3. **Mincer (1974) precedent**: The log-linear wage equation is the
   canonical specification in labor economics.

Specification:
    log(hearnw) = β₀ + β₁(educw) + β₂(experience) + β₃(experience²)
                + β₄(child6) + β₅(agew) + β₆(unemprate) + ε

Note: experience² is included to capture the concavity of the experience-earnings profile (returns to experience diminish over time).



## STEP 3.2: Baseline OLS with Robust Standard Errors

In [12]:
formula_baseline = ('log_hearnw ~ educw + experience + I(experience**2) '
                    '+ child6 + agew + unemprate')

model_baseline = smf.ols(formula=formula_baseline, data=df_work).fit(cov_type='HC1')

print("=" * 60)
print("BASELINE MODEL — OLS with HC1 Robust Standard Errors")
print("=" * 60)
print(model_baseline.summary())


BASELINE MODEL — OLS with HC1 Robust Standard Errors
                            OLS Regression Results                            
Dep. Variable:             log_hearnw   R-squared:                       0.158
Model:                            OLS   Adj. R-squared:                  0.146
Method:                 Least Squares   F-statistic:                     13.85
Date:                Wed, 18 Mar 2026   Prob (F-statistic):           2.20e-14
Time:                        11:56:04   Log-Likelihood:                -431.35
No. Observations:                 428   AIC:                             876.7
Df Residuals:                     421   BIC:                             905.1
Df Model:                           6                                         
Covariance Type:                  HC1                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------

## STEP 3.3: Interaction Term (educw × city)

#### Interaction Term: Education × City

**Economic hypothesis:** The return to education may be higher in urban
labor markets (city = 1) where demand for skilled workers is greater.
If β₇ > 0, an additional year of education yields a larger wage premium
in cities than in rural areas — consistent with agglomeration theory
(Moretti, 2004).

Specification:
    log(hearnw) = β₀ + β₁(educw) + β₂(experience) + β₃(experience²)
                + β₄(child6) + β₅(agew) + β₆(unemprate)
                + β₇(city) + β₈(educw × city) + ε

In [13]:
formula_interact = ('log_hearnw ~ educw * city + experience + I(experience**2) '
                    '+ child6 + agew + unemprate')

model_interact = smf.ols(formula=formula_interact, data=df_work).fit(cov_type='HC1')

print("=" * 60)
print("INTERACTION MODEL — educw × city (HC1 Robust SEs)")
print("=" * 60)
print(model_interact.summary())

INTERACTION MODEL — educw × city (HC1 Robust SEs)
                            OLS Regression Results                            
Dep. Variable:             log_hearnw   R-squared:                       0.159
Model:                            OLS   Adj. R-squared:                  0.143
Method:                 Least Squares   F-statistic:                     10.88
Date:                Wed, 18 Mar 2026   Prob (F-statistic):           5.82e-14
Time:                        11:56:05   Log-Likelihood:                -430.97
No. Observations:                 428   AIC:                             879.9
Df Residuals:                     419   BIC:                             916.5
Df Model:                           8                                         
Covariance Type:                  HC1                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------

## STEP 3.4: Coefficient Interpretation

#### Coefficient Interpretation (Ceteris Paribus)

From the **baseline model**:

- **educw (β = 0.1091, p < 0.001)**: A one-year increase in education is
  associated with an approximately 10.91% increase in hourly wages, holding
  experience, age, children, and local unemployment constant. This is
  statistically significant and consistent with human capital theory.

- **experience (β = 0.0410, p = 0.007)**: Each additional year of labor
  market experience increases wages by approximately 4.10%, ceteris paribus.
  The negative coefficient on experience² (β = -0.0008) confirms diminishing
  returns — experience raises wages early in a career but the effect levels off.

- **child6 (β = -0.0575, p = 0.583)**: Having an additional child under age 6
  is associated with a 5.75% decrease in wages, ceteris paribus, though this
  effect is not statistically significant. The motherhood penalty may be
  operating primarily through the selection margin (fewer women with young
  children choose to work at all) rather than through wages directly.

- **unemprate (β = -0.0027, p = 0.780)**: A one percentage point increase in
  the county unemployment rate is associated with a 0.27% decrease in wages,
  ceteris paribus. This effect is not statistically significant, suggesting
  local labor market conditions do not strongly predict individual wages in
  this sample.

From the **interaction model**:

- **educw (β = 0.1068, p < 0.001)**: The return to education for rural women
  (city = 0) is 10.68% per year of education, nearly identical to the baseline.

- **city (β = 0.0488, p = 0.883)**: Urban women with zero years of education
  earn 4.88% more than rural women, but this effect is not statistically
  significant.

- **educw:city (β = 0.0009, p = 0.972)**: The additional return to education
  for urban women is essentially zero (0.09% per year). There is no evidence
  that urban labor markets reward education more than rural ones in this sample,
  contrary to the agglomeration hypothesis.

In [14]:
print("\n--- Baseline Model: Key Coefficients ---")
baseline_coefs = model_baseline.params[['educw', 'experience', 'child6', 'unemprate']]
baseline_pvals = model_baseline.pvalues[['educw', 'experience', 'child6', 'unemprate']]
for var in baseline_coefs.index:
    print(f"  {var}: β = {baseline_coefs[var]:.4f} | "
          f"Approx. effect: {baseline_coefs[var]*100:.2f}% | "
          f"p = {baseline_pvals[var]:.3f}")

print("\n--- Interaction Model: educw × city ---")
interact_coef = model_interact.params.get('educw:city', None)
interact_pval = model_interact.pvalues.get('educw:city', None)
if interact_coef is not None:
    print(f"  educw:city: β = {interact_coef:.4f} | "
          f"Additional urban return: {interact_coef*100:.2f}% per year of education | "
          f"p = {interact_pval:.3f}")


--- Baseline Model: Key Coefficients ---
  educw: β = 0.1091 | Approx. effect: 10.91% | p = 0.000
  experience: β = 0.0410 | Approx. effect: 4.10% | p = 0.007
  child6: β = -0.0575 | Approx. effect: -5.75% | p = 0.583
  unemprate: β = -0.0027 | Approx. effect: -0.27% | p = 0.780

--- Interaction Model: educw × city ---
  educw:city: β = 0.0009 | Additional urban return: 0.09% per year of education | p = 0.972


## STEP 3.5: Omitted Variable Bias Defense

#### Omitted Variable Bias (OVB) Defense

**Variable we wish we had: Cognitive ability / IQ score**

The most significant omitted variable is a direct measure of cognitive
ability. If more able women both (a) acquire more education and (b) earn
higher wages independently of education, then the OLS estimate of β₁ is
upward biased — we would be attributing to education some of the wage
premium that actually reflects innate ability.

Formally: Bias = (effect of ability on wages) × (correlation between
ability and education). Both terms are plausibly positive, so OLS
likely *overstates* the true causal return to education.

**Possible remedy (Phase 4):** An Instrumental Variable (IV) strategy
using parents' education (educwm, educwf) as instruments for educw —
these variables plausibly affect a woman's education but have no direct
effect on her wages, satisfying the exclusion restriction.

In [15]:
print("Phase 3 Checklist:")
print("  [✓] Baseline OLS using statsmodels (NOT sklearn)")
print("  [✓] HC1 Heteroskedasticity-Robust Standard Errors")
print("  [✓] Interaction term: educw × city")
print("  [✓] Coefficients interpreted in ceteris paribus language")
print("  [✓] Omitted Variable Bias defense prepared (cognitive ability)")

Phase 3 Checklist:
  [✓] Baseline OLS using statsmodels (NOT sklearn)
  [✓] HC1 Heteroskedasticity-Robust Standard Errors
  [✓] Interaction term: educw × city
  [✓] Coefficients interpreted in ceteris paribus language
  [✓] Omitted Variable Bias defense prepared (cognitive ability)
